In [ ]:
# cach 2: chẩun hóa nhiều hơn 
import os
import cv2
import numpy as np
from tqdm import tqdm

input_dir  = r"C:\Users\Admin\Documents\xu_ly_anh_khop\input"
output_dir = r"C:\Users\Admin\Documents\xu_ly_anh_khop\output_qroi8_3"
os.makedirs(output_dir, exist_ok=True)

TARGET_SIZE = 8
STANDARD_WIDTH = 64

# giam nhieu
def denoise(img):
    return cv2.GaussianBlur(img,(3,3),0)

# auto canny
def auto_canny(img, sigma=0.33):
    v = np.median(img)
    return cv2.Canny(img,
                     int(max(0,(1-sigma)*v)),
                     int(min(255,(1+sigma)*v)))

# chuan hoa sang som
def normalize_clahe(img):
    clahe = cv2.createCLAHE(2.0,(8,8))
    return clahe.apply(img)

# tim khe khop
def find_joint_line(gray):
    edges = auto_canny(gray)
    proj = np.sum(edges, axis=1)
    proj = cv2.GaussianBlur(proj.reshape(-1,1),(31,1),0).ravel()
    return np.argmax(proj)

# xoay ngang
def rotate_to_horizontal(gray):
    edges = auto_canny(gray)
    lines = cv2.HoughLines(edges,1,np.pi/180,120)

    if lines is None:
        return gray

    angles = []
    for rho,theta in lines[:,0]:
        deg = (theta - np.pi/2)*180/np.pi
        if abs(deg) < 30:
            angles.append(deg)

    if len(angles) == 0:
        return gray

    angle = np.median(angles)
    h,w = gray.shape
    M = cv2.getRotationMatrix2D((w//2,h//2), angle, 1.0)

    return cv2.warpAffine(gray, M, (w,h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)

# crop 2 vien xuong
def crop_joint_width(gray, y):
    edges = auto_canny(gray)

    band = int(gray.shape[0]*0.15)
    y1 = max(0, y-band)
    y2 = min(gray.shape[0], y+band)

    roi_band = edges[y1:y2,:]

    proj_x = np.sum(roi_band, axis=0)
    proj_x = cv2.GaussianBlur(proj_x.reshape(1,-1),(1,31),0).ravel()

    threshold = 0.2 * np.max(proj_x)
    xs = np.where(proj_x > threshold)[0]

    if len(xs) < 5:
        return gray, 0, gray.shape[1]

    x1 = xs[0]
    x2 = xs[-1]

    if x2 <= x1:
        return gray, 0, gray.shape[1]

    return gray[:, x1:x2], x1, x2

# scale theo chieu rong
def normalize_width(img, width):
    if width is None or width <= 1:
        width = img.shape[1]

    if width == 0:
        width = 1

    scale = STANDARD_WIDTH / width
    h = int(img.shape[0] * scale)

    if h <= 0:
        h = TARGET_SIZE

    return cv2.resize(img, (STANDARD_WIDTH, h),
                      interpolation=cv2.INTER_LINEAR)

# crop chieu cao
def crop_height_by_ratio(img):
    h,w = img.shape

    band_h = int(h * 0.6)
    y_center = h // 2

    y1 = max(0, y_center - band_h//2)
    y2 = min(h, y_center + band_h//2)

    if y2 <= y1:
        return img

    return img[y1:y2,:]

# resize giu thong tin
def resize_final(img):
    return cv2.resize(img,(TARGET_SIZE,TARGET_SIZE),
                      interpolation=cv2.INTER_AREA)

# MAIN
for root,_,files in os.walk(input_dir):
    for file in tqdm(files):

        if not file.lower().endswith((".png",".jpg",".jpeg")):
            continue

        path = os.path.join(root,file)
        img = cv2.imread(path,0)

        if img is None:
            continue

        # 1 giam nhieu
        img = denoise(img)

        # 2 chuan hoa sang som
        img = normalize_clahe(img)

        # 3 tim khe
        y = find_joint_line(img)

        # 4 xoay
        img = rotate_to_horizontal(img)

        # 5 crop ngang
        img, x1, x2 = crop_joint_width(img, y)

        width = x2 - x1
        if width <= 1:
            width = img.shape[1]

        # 6 scale
        img = normalize_width(img, width)

        # 7 crop doc
        img = crop_height_by_ratio(img)

        # 8 resize
        img = resize_final(img)

        # save
        rel = os.path.relpath(path,input_dir)
        save_path = os.path.join(output_dir,rel)

        os.makedirs(os.path.dirname(save_path),exist_ok=True)

        cv2.imwrite(save_path,img)

print("DONE")

0it [00:00, ?it/s]
0it [00:00, ?it/s]
100%|██████████| 206/206 [00:00<00:00, 269.77it/s]
0it [00:00, ?it/s]
100%|██████████| 206/206 [00:00<00:00, 247.04it/s]

DONE


In [2]:
#! py -3.14 -m pip install pennylane

In [3]:
import os
import cv2
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

DATA_DIR = r"C:\Users\Admin\Documents\xu_ly_anh_khop\output_qroi8_3"

def load_dataset(data_dir):
    X, y = [], []
    label_map = {
        "0Normal":0,
        "1Doubtful":1,
        "2Mild":2,
        "3Moderate":3,
        "4Severe":4
    }

    for root, dirs, files in os.walk(data_dir):
        for file in files:
            if not file.lower().endswith((".png",".jpg",".jpeg")):
                continue

            path = os.path.join(root, file)
            label_name = os.path.basename(os.path.dirname(path))
            if label_name not in label_map:
                continue

            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

            # chuẩn hóa + scale cho quantum (quan trọng)
            img = (img / 255.0) * np.pi

            X.append(img.flatten())  # 64 chiều
            y.append(label_map[label_name])

    return np.array(X), np.array(y)

X, y = load_dataset(DATA_DIR)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# mo hinh quantum
n_qubits = 8
n_layers = 8   # train ổn định hơn
dev = qml.device("default.qubit", wires=n_qubits)

def encode_layer(features):
    for i in range(n_qubits):
        qml.Hadamard(wires=i) # 
        qml.RY(features[i], wires=i)

def variational_layer(weights):
    for i in range(n_qubits):
        qml.RY(weights[i], wires=i)
        qml.RZ(weights[i], wires=i) # 

    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])

@qml.qnode(dev)
def quantum_circuit(x, weights):

    chunks = np.array_split(x, n_layers)

    for i in range(n_layers):
        chunk = chunks[i]

        # nếu thiếu thì pad thêm
        if len(chunk) < n_qubits:
            chunk = np.pad(chunk, (0, n_qubits - len(chunk)))

        encode_layer(chunk[:n_qubits])
        variational_layer(weights[i])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


# CLASSICAL PART

n_classes = 5

weights = pnp.random.randn(n_layers, n_qubits, requires_grad=True)
W_classical = pnp.random.randn(n_qubits, n_classes, requires_grad=True)

def softmax(x):
    e = pnp.exp(x - pnp.max(x))  # tránh overflow
    return e / pnp.sum(e)

def forward(x, weights, W_classical):
    q_out = pnp.array(quantum_circuit(x, weights))
    logits = pnp.dot(q_out, W_classical)
    return softmax(logits)

def loss_fn(weights, W_classical, X, y):
    loss = 0
    for i in range(len(X)):
        probs = forward(X[i], weights, W_classical)
        loss -= pnp.log(probs[y[i]] + 1e-9)
    return loss / len(X)

n_classes = 5

weights = pnp.random.randn(n_layers, n_qubits, requires_grad=True)
W_classical = pnp.random.randn(n_qubits, n_classes, requires_grad=True)


In [5]:
# TRAIN (MINI-BATCH)
opt = qml.AdamOptimizer(0.01)
epochs = 10
batch_size = 32

for epoch in range(epochs):

    # shuffle
    idx = np.random.permutation(len(X_train))

    for i in range(0, len(X_train), batch_size):
        batch_idx = idx[i:i+batch_size]
        X_batch = X_train[batch_idx]
        y_batch = y_train[batch_idx]

        weights, W_classical = opt.step(
            lambda w,c: loss_fn(w,c,X_batch,y_batch),
            weights, W_classical
        )

    # tính loss nhanh trên 100 sample
    train_loss = loss_fn(weights, W_classical, X_train[:100], y_train[:100])
    print(f"Epoch {epoch+1} | Loss: {train_loss:.4f}")


def predict(X):
    preds = []
    for i in range(len(X)):
        probs = forward(X[i], weights, W_classical)
        preds.append(int(np.argmax(probs)))
    return np.array(preds)

y_pred = predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Test Accuracy:", acc)


Epoch 1 | Loss: 1.5264
Epoch 2 | Loss: 1.4836
Epoch 3 | Loss: 1.4779
Epoch 4 | Loss: 1.4531
Epoch 5 | Loss: 1.4508
Epoch 6 | Loss: 1.4227
Epoch 7 | Loss: 1.4259
Epoch 8 | Loss: 1.4108
Epoch 9 | Loss: 1.4191
Epoch 10 | Loss: 1.4223
Test Accuracy: 0.37272727272727274
